# Notebook 1 — Preprocessing
**What this notebook does:** Extract frames from MSR-VTT and MSVD videos, run frozen CLIP to produce visual embeddings, and save everything to Google Drive organised by split (train / val / test).

**Runtime:** T4 GPU (free tier)

## Split strategy

### MSR-VTT
Splits come from the `split` field **inside `MSRVTT_data.json`** — no CSV files needed.
- `split == 'train'` → train set (~6,513 videos)
- `split == 'validate'` → val set (~497 videos)
- `split == 'test'` → test set (~2,990 videos)

### MSVD
Splits come from the three `.txt` files in `raw_data/` — these ARE needed because
`AllVideoDescriptions.txt` has no split field.
- `train_list.txt` → train set (~1,200 videos)
- `val_list.txt` → val set (~100 videos)
- `test_list.txt` → test set (~670 videos)

## Outputs to Drive
```
ariel/embeddings/
  msrvtt/
    train/manifest.json  +  {video_id}.pt files
    val/manifest.json    +  {video_id}.pt files
    test/manifest.json   +  {video_id}.pt files
  msvd/
    train/manifest.json  +  {video_id}.pt files
    val/manifest.json    +  {video_id}.pt files
    test/manifest.json   +  {video_id}.pt files
```
NB3 loads train + val manifests. NB4 loads test manifests only.

---

In [1]:
!pip install -q transformers opencv-python-headless tqdm

## Step 1 · Mount Drive and verify structure

Before running, make sure your Drive looks like this:
```
My Drive/
  MSRVTT/
    raw_data/
      MSRVTT_data.json          ← contains all videos + captions + split labels
    raw_videos/
      video0.mp4 … video9999.mp4
  MSVD/
    raw_data/
      AllVideoDescriptions.txt  ← tab-separated: video_id, start, end, lang, caption
      train_list.txt            ← one video_id per line
      val_list.txt
      test_list.txt
    raw_videos/
      _0nX-El-ySo_83_93.avi …
  ariel/
    model.py                    ← uploaded manually
```

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ── Paths — edit only if your Drive folders are named differently ──
MSRVTT_ROOT = '/content/drive/Shareddrives/DATA 298A/DATA/MSRVTT'
MSVD_ROOT   = '/content/drive/Shareddrives/DATA 298A/DATA/MSVD'
PROJECT     = '/content/drive/Shareddrives/DATA 298A/ariel'
OUTPUT_DIR  = f'{PROJECT}/embeddings'

# Create output split folders
for ds in ['msrvtt', 'msvd']:
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{OUTPUT_DIR}/{ds}/{split}', exist_ok=True)

# Verify key files exist before starting
checks = [
    f'{MSRVTT_ROOT}/raw_data/MSRVTT_data.json',
    f'{MSRVTT_ROOT}/raw_videos',
    f'{MSVD_ROOT}/raw_data/AllVideoDescriptions.txt',
    f'{MSVD_ROOT}/raw_data/train_list.txt',
    f'{MSVD_ROOT}/raw_data/val_list.txt',
    f'{MSVD_ROOT}/raw_data/test_list.txt',
    f'{MSVD_ROOT}/raw_videos',
    f'{PROJECT}/model.py',
]
all_ok = True
for path in checks:
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING"}  {path}')
    if not exists:
        all_ok = False

print('\nAll files found — ready to proceed.' if all_ok else '\nFix missing paths before continuing.')

Mounted at /content/drive
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSRVTT/raw_data/MSRVTT_data.json
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSRVTT/raw_videos
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_data/AllVideoDescriptions.txt
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_data/train_list.txt
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_data/val_list.txt
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_data/test_list.txt
  OK  /content/drive/Shareddrives/DATA 298A/DATA/MSVD/raw_videos
  OK  /content/drive/Shareddrives/DATA 298A/ariel/model.py

All files found — ready to proceed.


## Step 2 · Config

In [3]:
CONFIG = {
    'clip_model' : 'openai/clip-vit-large-patch14',
    'num_frames' : 8,    # frames sampled per video
    'frame_size' : 224,
    'batch_size' : 32,   # frames per CLIP forward pass
    'max_videos' : None, # set e.g. 100 for a quick smoke-test; None = process all
}
print(CONFIG)

{'clip_model': 'openai/clip-vit-large-patch14', 'num_frames': 8, 'frame_size': 224, 'batch_size': 32, 'max_videos': None}


## Step 3 · Load frozen CLIP vision encoder

In [4]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

clip_model     = CLIPModel.from_pretrained(CONFIG['clip_model']).to(device)
clip_processor = CLIPProcessor.from_pretrained(CONFIG['clip_model'])

# Freeze — used for inference only, weights never change
for p in clip_model.parameters():
    p.requires_grad = False
clip_model.eval()
print('CLIP loaded and frozen.')

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded and frozen.


## Step 4 · Frame extraction and embedding helpers

In [8]:
import cv2
import numpy as np
from PIL import Image


def extract_frames(video_path, num_frames=8, size=224):
    """Sample num_frames evenly-spaced frames. Returns (frames, timestamps, duration)."""
    cap      = cv2.VideoCapture(video_path)
    total    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25.0
    duration = total / fps
    indices  = np.linspace(0, max(total - 1, 0), num_frames, dtype=int)

    frames, timestamps = [], []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size, size))
        frames.append(Image.fromarray(frame))
        timestamps.append(round(idx / fps, 3))

    cap.release()
    return frames, timestamps, round(duration, 3)


def embed_frames(frames):
    """Run frozen CLIP on PIL frames. Returns (num_frames, 768) normalised tensor."""
    all_embeds = []
    for i in range(0, len(frames), CONFIG['batch_size']):
        batch  = frames[i : i + CONFIG['batch_size']]
        inputs = clip_processor(images=batch, return_tensors='pt').to(device)
        with torch.no_grad():
            # vision_model returns BaseModelOutputWithPooling in transformers 5.x
            # use clip_model.vision_model + projection instead of get_image_features
            vision_outputs = clip_model.vision_model(**inputs)
            pooled = vision_outputs.pooler_output          # (B, hidden_dim)
            emb    = clip_model.visual_projection(pooled)  # (B, 768)
            emb    = emb / emb.norm(dim=-1, keepdim=True)  # L2 normalise
        all_embeds.append(emb.cpu())
    return torch.cat(all_embeds, dim=0)  # (num_frames, 768)


def process_video(video_path, captions):
    """
    Extract frames + embeddings for one video.
    Returns a dict ready to torch.save(), or None if the video can't be read.
    """
    frames, timestamps, duration = extract_frames(
        video_path, CONFIG['num_frames'], CONFIG['frame_size'])
    if not frames:
        return None
    return {
        'visual_embeds': embed_frames(frames),  # (8, 768)
        'timestamps'   : timestamps,
        'duration'     : duration,
        'captions'     : captions,
    }


print('Helpers defined.')

Helpers defined.


## Step 5 · Process MSR-VTT

`MSRVTT_data.json` structure:
```json
{
  "videos":    [{"video_id": "video0", "split": "train", ...}, ...],
  "sentences": [{"video_id": "video0", "caption": "..."},     ...]
}
```
The `split` field values are `"train"`, `"validate"`, and `"test"`.
We map `"validate"` → `"val"` to keep naming consistent with MSVD.

In [9]:
import json, csv
from pathlib import Path
from tqdm import tqdm


def load_video_ids_from_csv(csv_path):
    """Read video IDs from an MSR-VTT split CSV. Returns a set of video_id strings."""
    ids = set()
    with open(csv_path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # column is named 'video_id' in both train and test CSVs
            vid_id = row.get('video_id', row.get('videoid', '')).strip()
            if vid_id:
                ids.add(vid_id)
    return ids


def process_msrvtt():
    print('Loading MSRVTT_data.json...')
    with open(f'{MSRVTT_ROOT}/raw_data/MSRVTT_data.json') as f:
        data = json.load(f)

    # Build video_id → captions mapping from the sentences list
    video_captions = {}
    for s in data['sentences']:
        video_captions.setdefault(s['video_id'], []).append(s['caption'])

    # Load split IDs from the CSV files
    # train_9k is the standard training split used in retrieval papers
    train_ids = load_video_ids_from_csv(f'{MSRVTT_ROOT}/raw_data/MSRVTT_train.9k.csv')
    test_ids  = load_video_ids_from_csv(f'{MSRVTT_ROOT}/raw_data/MSRVTT_JSFUSION_test.csv')

    # val = everything not in train or test
    all_ids   = set(v['video_id'] for v in data['videos'])
    val_ids   = all_ids - train_ids - test_ids

    print(f'Split counts from CSVs:')
    print(f'  train : {len(train_ids)}')
    print(f'  val   : {len(val_ids)}')
    print(f'  test  : {len(test_ids)}')

    def get_split(vid_id):
        if vid_id in train_ids: return 'train'
        if vid_id in test_ids:  return 'test'
        if vid_id in val_ids:   return 'val'
        return 'train'  # fallback

    videos = data['videos']
    if CONFIG['max_videos']:
        videos = videos[:CONFIG['max_videos']]

    manifests = {'train': [], 'val': [], 'test': []}
    skipped = 0

    for v in tqdm(videos, desc='MSR-VTT'):
        vid_id     = v['video_id']
        split      = get_split(vid_id)
        video_path = f'{MSRVTT_ROOT}/raw_videos/{vid_id}.mp4'
        embed_path = Path(f'{OUTPUT_DIR}/msrvtt/{split}/{vid_id}.pt')
        captions   = video_captions.get(vid_id, [])

        if not os.path.exists(video_path):
            skipped += 1
            continue

        if not embed_path.exists():
            result = process_video(video_path, captions)
            if result is None:
                skipped += 1
                continue
            torch.save(result, embed_path)

        manifests[split].append({
            'video_id'  : vid_id,
            'embed_path': str(embed_path),
            'captions'  : captions,
            'split'     : split,
        })

    for split, records in manifests.items():
        manifest_path = f'{OUTPUT_DIR}/msrvtt/{split}/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(records, f, indent=2)
        print(f'  MSR-VTT {split:5s}: {len(records):5d} videos → {manifest_path}')

    print(f'  Skipped: {skipped}')
    return manifests


msrvtt_manifests = process_msrvtt()

Loading MSRVTT_data.json...
Split counts from CSVs:
  train : 9000
  val   : 0
  test  : 1000


MSR-VTT: 100%|██████████| 10000/10000 [3:55:48<00:00,  1.41s/it]


  MSR-VTT train:  9000 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msrvtt/train/manifest.json
  MSR-VTT val  :     0 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msrvtt/val/manifest.json
  MSR-VTT test :  1000 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msrvtt/test/manifest.json
  Skipped: 0


## Step 6 · Process MSVD

`AllVideoDescriptions.txt` is tab-separated with columns:
```
video_id  \t  start_time  \t  end_time  \t  language  \t  caption
```
The video_id matches the `.avi` filename stem (e.g. `_0nX-El-ySo_83_93`).

`train_list.txt`, `val_list.txt`, `test_list.txt` each contain one video_id per line.
These are the only source of split information for MSVD.

In [10]:
def load_id_list(path):
    """Read a split list txt file → set of video IDs."""
    with open(path) as f:
        return set(line.strip() for line in f if line.strip())


def parse_msvd_captions():
    """
    Parse AllVideoDescriptions.txt.
    Returns dict: {video_id: [caption, caption, ...]}
    Only keeps English captions (language field == 'English').
    """
    captions = {}
    desc_path = f'{MSVD_ROOT}/raw_data/AllVideoDescriptions.txt'

    with open(desc_path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 5:
                continue  # skip malformed lines
            vid_id   = parts[0].strip()
            language = parts[3].strip()
            caption  = parts[4].strip()
            if language == 'English' and caption:
                captions.setdefault(vid_id, []).append(caption)

    print(f'MSVD: loaded captions for {len(captions)} unique video IDs')
    return captions


def process_msvd():
    # Load split lists — these define the official splits
    train_ids = load_id_list(f'{MSVD_ROOT}/raw_data/train_list.txt')
    val_ids   = load_id_list(f'{MSVD_ROOT}/raw_data/val_list.txt')
    test_ids  = load_id_list(f'{MSVD_ROOT}/raw_data/test_list.txt')
    print(f'Split sizes — train: {len(train_ids)}, val: {len(val_ids)}, test: {len(test_ids)}')

    msvd_captions = parse_msvd_captions()

    video_files = sorted(Path(f'{MSVD_ROOT}/raw_videos').glob('*.avi'))
    if CONFIG['max_videos']:
        video_files = video_files[:CONFIG['max_videos']]

    def get_split(vid_id):
        if vid_id in train_ids: return 'train'
        if vid_id in val_ids:   return 'val'
        if vid_id in test_ids:  return 'test'
        return None  # video not in any split list — skip

    manifests = {'train': [], 'val': [], 'test': []}
    skipped = 0
    no_split = 0

    for vf in tqdm(video_files, desc='MSVD'):
        vid_id = vf.stem
        split  = get_split(vid_id)

        if split is None:
            no_split += 1
            continue  # video not in any official split

        captions   = msvd_captions.get(vid_id, [])
        embed_path = Path(f'{OUTPUT_DIR}/msvd/{split}/{vid_id}.pt')

        if not embed_path.exists():
            result = process_video(str(vf), captions)
            if result is None:
                skipped += 1
                continue
            torch.save(result, embed_path)

        manifests[split].append({
            'video_id'  : vid_id,
            'embed_path': str(embed_path),
            'captions'  : captions,
            'split'     : split,
        })

    for split, records in manifests.items():
        manifest_path = f'{OUTPUT_DIR}/msvd/{split}/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(records, f, indent=2)
        print(f'  MSVD {split:5s}: {len(records):5d} videos → {manifest_path}')

    print(f'  Skipped (bad video): {skipped} | Not in any split: {no_split}')
    return manifests


msvd_manifests = process_msvd()

Split sizes — train: 1200, val: 100, test: 670
MSVD: loaded captions for 0 unique video IDs


MSVD: 100%|██████████| 1970/1970 [28:41<00:00,  1.14it/s]

  MSVD train:  1200 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msvd/train/manifest.json
  MSVD val  :   100 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msvd/val/manifest.json
  MSVD test :   670 videos → /content/drive/Shareddrives/DATA 298A/ariel/embeddings/msvd/test/manifest.json
  Skipped (bad video): 0 | Not in any split: 0


## Step 7 · Summary and sanity check

In [14]:
print('=== Final split summary ===')
print()
total = {'train': 0, 'val': 0, 'test': 0}

for ds, manifests in [('MSR-VTT', msrvtt_manifests), ('MSVD', msvd_manifests)]:
    print(f'{ds}:')
    for split in ['train', 'val', 'test']:
        n = len(manifests[split])
        total[split] += n
        print(f'  {split:5s}: {n:5d} videos')
    print()

print('Combined totals (what NB3 and NB4 will use):')
for split in ['train', 'val', 'test']:
    print(f'  {split:5s}: {total[split]:5d} videos')

print()
print('NB3 uses: train/ + val/ manifests from both datasets')
print('NB4 uses: test/  manifests from both datasets')

# Spot-check one embedding
print()
print('Spot-checking one MSR-VTT train embedding...')
if msrvtt_manifests['train']:
    sample = torch.load(msrvtt_manifests['train'][0]['embed_path'], weights_only=False)
    print(f'  visual_embeds shape : {sample["visual_embeds"].shape}')  # (8, 768)
    print(f'  timestamps          : {sample["timestamps"]}')
    print(f'  num captions        : {len(sample["captions"])}')
    print(f'  sample caption      : {sample["captions"][0]}')
    assert sample['visual_embeds'].shape == (8, 768), 'Unexpected shape!'
    print('Shape correct. NB1 complete — move to NB2.')

=== Final split summary ===

MSR-VTT:
  train:  9000 videos
  val  :     0 videos
  test :  1000 videos

MSVD:
  train:  1200 videos
  val  :   100 videos
  test :   670 videos

Combined totals (what NB3 and NB4 will use):
  train: 10200 videos
  val  :   100 videos
  test :  1670 videos

NB3 uses: train/ + val/ manifests from both datasets
NB4 uses: test/  manifests from both datasets

Spot-checking one MSR-VTT train embedding...
  visual_embeds shape : torch.Size([8, 768])
  timestamps          : [np.float64(0.0), np.float64(1.667), np.float64(3.333), np.float64(5.0), np.float64(7.0), np.float64(8.667), np.float64(10.333), np.float64(12.333)]
  num captions        : 20
  sample caption      : a car is shown
Shape correct. NB1 complete — move to NB2.
